# Customer Support AI — Multi-layered Defense Pipeline

**Author:** Burcu Tatlı (AI Operations)  
**Date:** April 2026  
**Repo:** [customer-support-ai](https://github.com/burcutatli/customer-support-ai)

## What this demo does

This notebook demonstrates a customer support AI with **defense-in-depth architecture**:

```
Customer Message
       ↓
[Layer 1] Escalation Filter (keyword + behavioral)
       ↓
[Layer 2] Haiku Classifier (with confidence threshold)
       ↓
[Layer 3] RAG Pipeline (Sonnet + Chroma vector DB)
       ↓
Customer Response (or escalation)
```

Each layer can independently route to human escalation. The customer never receives an unsafe answer.

## How to run

1. **Add your Anthropic API key** to Colab Secrets (key icon in left sidebar) with name `ANTHROPIC_API_KEY`
2. Run cells top to bottom
3. Try your own messages in the last cell

## Step 1 — Install dependencies

In [ ]:
!pip install -q anthropic chromadb

## Step 2 — Set up project structure and write all files

This cell creates the full project structure and writes:
- 6 markdown KB files (shipping, billing, account, product, sales, order_change)
- 1 JSON file with 9 escalation categories and 50+ keywords
- 5 Python source files

Total: ~600 lines of code + ~2,000 words of KB content.

In [ ]:
import os
from pathlib import Path

# Create project structure
ROOT = Path("/content/customer-support-ai")
for subdir in ["src", "config", "knowledge_base", "tests", "logs"]:
    (ROOT / subdir).mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print("Subdirectories created:")
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print(f"  - {p.name}/")

### Write the knowledge base markdown files

In [ ]:
# Shipping KB
(ROOT / "knowledge_base" / "shipping.md").write_text("""# Shipping & Delivery

## When this document is used

The customer is asking about:
- Order delivery times
- Tracking links
- Address change requests

**Keywords:** shipping, delivery, tracking, where is my order, address change, when will it arrive, ship date, courier

## Standard Answers

### Q1: When will my order arrive?

Hello! Our standard delivery time is **2-5 business days**. Once your order ships, you'll receive a tracking link via email. The tracking link becomes active **within 24 hours** after dispatch.

If your order hasn't arrived after 5 business days, please share your order number and we'll investigate immediately.

### Q2: My tracking link isn't working

Tracking links activate in the courier's system **within 24 hours** of dispatch. So if you click immediately after shipping, it's normal that it doesn't work yet.

If 24 hours have passed and you're still having trouble, please share your **order number** so we can contact the courier directly.

### Q3: Can I change my shipping address?

This depends on your order's current status:

- **If your order has not yet shipped:** Address change is possible. Share your order number and we'll update it right away.

- **If your order has already shipped:** Address change is **not guaranteed** — it depends on the courier's system. We'll attempt to redirect for you. Please share your order number.

## Edge Cases

- **"It's been more than 5 days"** → Request order number, escalate to logistics team
- **"International shipping"** → Not in this KB, say "Our specialized team will contact you for international orders" and escalate
- **"Same-day delivery"** → This service doesn't exist yet, say "We don't currently offer same-day delivery"
- **"Lost package"** → Always escalate to human

## Internal Notes (NOT shown to customer)

- 2-5 business days target, can extend to 7 days during peak
- Same-day delivery is being evaluated for Q4 2026 — don't promise
- Logistics partners: FedEx (primary), USPS (secondary)
""", encoding="utf-8")

# Billing KB
(ROOT / "knowledge_base" / "billing.md").write_text("""# Billing & Payments

## When this document is used

The customer is asking about:
- Invoice download issues
- Promo codes / discount codes
- Payment history queries

**Keywords:** invoice, payment, promo code, discount code, receipt, billing, charge, transaction

## Standard Answers

### Q1: How do I download my invoice?

You can download your invoice from your **user dashboard** by following these steps:

1. Log into your account
2. Go to the **"My Orders"** section
3. Click the **"Download Invoice"** button next to the relevant order

If you can't find or download your invoice, please share your order number and we'll help you out.

### Q2: My promo code isn't working

Promo codes are usually **single-use** and tied to **expiration dates**. There are a few common reasons:

- The code may have **expired**
- The code may have been **used before**
- The code may only be valid for **certain products or minimum cart values**

Could you share the code with us? We'll check the conditions it was issued under.

## Internal Notes (NOT shown to customer)

- Double charge cases: finance team responds within 3 business days max
- Promo codes above 20% off are usually single-use only
- Corporate invoicing redirect to: finance@lumio.com
""", encoding="utf-8")

# Account KB
(ROOT / "knowledge_base" / "account.md").write_text("""# Account Management

## When this document is used

The customer is asking about:
- Password reset, "I forgot my password"
- Login problems
- Account access issues

**Keywords:** password, login, sign in, can't log in, account, membership, forgot password, locked out

## Standard Answers

### Q1: How do I reset my password?

To reset your password:

1. On the login page, click **"Forgot Password"**
2. Enter your registered email address
3. Click the reset link in the email and set a new password

**Important:** The reset email sometimes lands in your **spam/junk folder**. If you don't see it in your inbox, please check there first.

If you're still having trouble, please reply and we'll help you out.

### Q2: I can't log in, what should I do?

Let's troubleshoot together. First, please check:

- Are you entering your password correctly? **Watch for case sensitivity**
- Is your **Caps Lock** off?
- Try clearing your browser's **cookies/cache** — this often resolves login issues

If the problem persists, please share:
- Your **device** (laptop, phone?)
- Your **browser** (Chrome, Safari, etc.)
- A **screenshot** of the error you're seeing

With these, our technical team can help you faster.
""", encoding="utf-8")

# Product KB
(ROOT / "knowledge_base" / "product.md").write_text("""# Product Information

## When this document is used

The customer is asking about:
- Product features, sizing, fit
- Stock status, restock dates
- Product specifications

**Keywords:** size, sizing, fit, stock, in stock, out of stock, when available, color, feature, specs

## Standard Answers

### Q1: What sizes does this product come in?

Detailed sizing and fit information is available **on each product page**. If you can't find what you're looking for there, please share the product name or link, and we'll check for you.

### Q2: When will this product be back in stock?

Our restock dates can sometimes change, so we **can't give a definite date**. But we have a couple of options for you:

- Use the **"Notify Me"** feature on the product page to receive an email when it's back in stock
- We can suggest similar alternative products if you'd like

Which would you prefer?

## Internal Notes (NOT shown to customer)

- Restock cycles average 2-4 weeks but supplier delays happen
- 'Notify Me' feature: ~200 notifications go out per day
""", encoding="utf-8")

# Sales KB
(ROOT / "knowledge_base" / "sales.md").write_text("""# Sales — Corporate & Bulk Orders

## When this document is used

The customer is asking about:
- Corporate offers, B2B inquiries
- Bulk order discounts
- Custom project quotes

**Keywords:** corporate, business, bulk, wholesale, b2b, discount, quote, enterprise, volume order

## Standard Answers

### Q1: Do you offer bulk discounts?

Yes, we can offer **special pricing** for orders above certain quantities. However, since pricing depends on quantity and product mix, it's better for us to prepare a custom quote for you.

Could you share a few details?
- Which products or category interests you?
- Approximate quantity you're considering?
- Any specific delivery timeline?

With this info, our sales team will get back to you within 1-2 business days.

### Q2: I want to get a corporate quote

For corporate sales, we have a **dedicated sales team**. To prepare a custom quote, we'll need a few details:

- Your company name and industry
- Product/service of interest
- Approximate user count or quantity
- Budget range (optional but helpful)

Share these and our sales team will reach out directly.

## Internal Notes (NOT shown to customer)

- Corporate sales team: sales@lumio.com
- 10+ units triggers special pricing
- 50+ units typically gets 10-15% discount — AI never mentions this
""", encoding="utf-8")

# Order Change KB
(ROOT / "knowledge_base" / "order_change.md").write_text("""# Order Change

## When this document is used

The customer wants to modify their order after placing it:
- Product change (different color, different size)
- Quantity change (add or remove items)
- Wrong order — completely changing it

**Keywords:** change order, modify order, wrong order, different product, add item, remove item, update order

## Standard Answers

### Q1: Can I change my order?

Order changes depend on **what stage your order is at**:

- **If your order has not yet shipped:** Changes are usually possible. Share your order number and we'll check right away.

- **If your order has already shipped:** Changes are usually not possible. In this case, you may need to receive the order and start a return process. Our team will find the best solution for you.

Could you share your order number?

### Q2: I ordered the wrong product

Don't worry, this happens often. Your options depend on your order status:

- **If not shipped yet:** We can either cancel the order and create a new one, or modify the product directly.

- **If already shipped:** You can use our **return process** after receiving the order to make changes.

Either way, just share your **order number** and our team will help you.
""", encoding="utf-8")

print("✓ 6 KB markdown files written")

### Write the escalation keywords JSON

In [ ]:
import json

escalation_keywords = {
    "metadata": {
        "version": "v1.0",
        "created_date": "2026-04-25",
        "created_by": "Burcu Tatli - AI Operations",
        "language": "en"
    },
    "escalation_keywords": {
        "refund": {
            "priority": "high",
            "destination_team": "refund_team",
            "keywords": ["refund", "return money", "money back", "want my money", "give me back", "reimburse", "return"]
        },
        "cancel": {
            "priority": "high",
            "destination_team": "support_lead",
            "keywords": ["cancel", "cancellation", "cancel order", "cancel my", "stop my order"]
        },
        "legal": {
            "priority": "critical",
            "destination_team": "legal_team",
            "keywords": ["lawyer", "attorney", "legal", "court", "sue", "lawsuit", "litigation"]
        },
        "complaint": {
            "priority": "high",
            "destination_team": "customer_satisfaction",
            "keywords": ["complaint", "terrible", "awful", "horrible", "worst", "bad service", "disappointed"]
        },
        "human_request": {
            "priority": "high",
            "destination_team": "support_general",
            "keywords": ["human", "agent", "real person", "operator", "live support", "speak to someone", "talk to a person"]
        },
        "anger": {
            "priority": "high",
            "destination_team": "senior_support",
            "keywords": ["angry", "furious", "pissed", "outraged", "fed up", "ridiculous"]
        },
        "double_charge": {
            "priority": "high",
            "destination_team": "finance_team",
            "keywords": ["double charge", "charged twice", "duplicate payment", "two payments", "billed twice"]
        },
        "delete_account": {
            "priority": "high",
            "destination_team": "privacy_team",
            "keywords": ["delete account", "close account", "remove account", "delete my data", "gdpr deletion"]
        },
        "damaged": {
            "priority": "high",
            "destination_team": "logistics_team",
            "keywords": ["damaged", "broken", "defective", "not working", "missing parts", "arrived broken"]
        }
    },
    "behavioral_triggers": {
        "rules": [
            {"name": "all_caps_shouting", "priority": "medium", "destination_team": "senior_support"},
            {"name": "multiple_exclamations", "priority": "medium", "destination_team": "senior_support"}
        ]
    }
}

(ROOT / "config" / "escalation_keywords.json").write_text(
    json.dumps(escalation_keywords, indent=2),
    encoding="utf-8"
)

print("✓ Escalation keywords JSON written")
print(f"  Total categories: {len(escalation_keywords['escalation_keywords'])}")
total_kw = sum(len(c['keywords']) for c in escalation_keywords['escalation_keywords'].values())
print(f"  Total keywords: {total_kw}")

## Step 3 — Set up the API key

Add your `ANTHROPIC_API_KEY` to **Colab Secrets** first (key icon in left sidebar).  
Then run this cell to load it.

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("✓ API key loaded" if os.environ.get("ANTHROPIC_API_KEY") else "✗ API key NOT set")

## Step 4 — The full AI pipeline (all 5 modules in one cell)

In a real project, these would be separate `.py` files.  
For Colab convenience, everything is in one cell.

In [ ]:
import json
import logging
import re
import sys
from pathlib import Path
from anthropic import Anthropic, APIError
import chromadb

# ============================================================
# CONFIG (would be config.py)
# ============================================================

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
CLASSIFIER_MODEL = "claude-haiku-4-5"
RESPONDER_MODEL = "claude-sonnet-4-5"
CLASSIFIER_MAX_TOKENS = 100
RESPONDER_MAX_TOKENS = 500
TOP_K_RETRIEVAL = 3
MIN_CONFIDENCE_TO_ANSWER = 0.50

KB_DIRECTORY = ROOT / "knowledge_base"
ESCALATION_KEYWORDS_PATH = ROOT / "config" / "escalation_keywords.json"

VALID_CATEGORIES = [
    "shipping", "billing", "account", "product",
    "sales", "order_change", "general"
]

ESCALATION_RESPONSE_TEMPLATE = (
    "I understand your concern. Let me connect you with a team member "
    "who can help you better. They'll get back to you shortly."
)

# Configure simple logging
logging.basicConfig(
    level="INFO",
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)


# ============================================================
# ESCALATION FILTER (would be escalation_filter.py)
# ============================================================

class EscalationFilter:
    def __init__(self, keywords_path):
        with open(keywords_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.keyword_categories = data.get("escalation_keywords", {})
        self.behavioral_rules = data.get("behavioral_triggers", {}).get("rules", [])
    
    def check(self, message):
        message_lower = message.lower().strip()
        
        for category_name, category_data in self.keyword_categories.items():
            keywords = category_data.get("keywords", [])
            destination_team = category_data.get("destination_team")
            for keyword in keywords:
                if keyword.lower() in message_lower:
                    return (True, f"{category_name}_keyword", destination_team)
        
        for rule in self.behavioral_rules:
            rule_name = rule.get("name")
            destination_team = rule.get("destination_team")
            if rule_name == "all_caps_shouting" and self._is_shouting(message):
                return (True, "all_caps_shouting", destination_team)
            elif rule_name == "multiple_exclamations" and self._has_excessive_punctuation(message):
                return (True, "multiple_exclamations", destination_team)
        
        return (False, None, None)
    
    @staticmethod
    def _is_shouting(message):
        if len(message) <= 10:
            return False
        letters = [c for c in message if c.isalpha()]
        if not letters:
            return False
        return sum(1 for c in letters if c.isupper()) / len(letters) > 0.70
    
    @staticmethod
    def _has_excessive_punctuation(message):
        return message.count("!") + message.count("?") >= 3


# ============================================================
# CLASSIFIER (would be classifier.py)
# ============================================================

class MessageClassifier:
    def __init__(self, api_key, model, max_tokens, valid_categories):
        self.client = Anthropic(api_key=api_key)
        self.model = model
        self.max_tokens = max_tokens
        self.valid_categories = valid_categories
    
    def classify(self, message):
        category_list = "\n".join(f"- {c}" for c in self.valid_categories)
        system_prompt = f"""You are a customer support message classifier.
Categorize the message into ONE of these categories:

{category_list}

Respond ONLY with valid JSON:
{{"category": "shipping", "confidence": 0.92, "reasoning": "brief explanation"}}

Rules:
- category MUST be lowercase, exactly from the list
- confidence between 0.0 and 1.0
- reasoning max 15 words
- NO markdown, NO code blocks, NO extra text"""
        
        try:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=self.max_tokens,
                system=system_prompt,
                messages=[{"role": "user", "content": message}]
            )
            raw = response.content[0].text.strip()
            
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()
            
            parsed = json.loads(raw)
            category = parsed.get("category", "").lower()
            confidence = float(parsed.get("confidence", 0.0))
            reasoning = parsed.get("reasoning", "")
            
            if category not in self.valid_categories:
                return {"category": "general", "confidence": 0.0, "reasoning": "invalid_category"}
            
            confidence = max(0.0, min(1.0, confidence))
            return {"category": category, "confidence": confidence, "reasoning": reasoning}
        
        except Exception as e:
            logger.error(f"Classifier error: {e}")
            return {"category": "general", "confidence": 0.0, "reasoning": f"error: {e}"}


# ============================================================
# RAG PIPELINE (would be rag_pipeline.py)
# ============================================================

class RAGPipeline:
    def __init__(self, kb_directory, anthropic_api_key, model, max_tokens, top_k):
        self.kb_directory = kb_directory
        self.client = Anthropic(api_key=anthropic_api_key)
        self.model = model
        self.max_tokens = max_tokens
        self.top_k = top_k
        self.chroma_client = chromadb.Client()
        try:
            self.chroma_client.delete_collection(name="customer_support_kb")
        except Exception:
            pass
        self.collection = self.chroma_client.create_collection(name="customer_support_kb")
    
    def index_knowledge_base(self):
        md_files = list(self.kb_directory.glob("*.md"))
        all_chunks, all_metadatas, all_ids = [], [], []
        
        for md_file in md_files:
            content = md_file.read_text(encoding="utf-8")
            sections = re.split(r"^(## .+)$", content, flags=re.MULTILINE)
            
            intro = sections[0].strip()
            if intro:
                title_match = re.search(r"^# (.+)$", intro, flags=re.MULTILINE)
                title = title_match.group(1) if title_match else md_file.stem
                all_chunks.append(intro)
                all_metadatas.append({"source_file": md_file.name, "section": f"Intro: {title}"})
                all_ids.append(f"{md_file.stem}_intro")
            
            for i in range(1, len(sections), 2):
                if i + 1 < len(sections):
                    header = sections[i].strip()
                    body = sections[i + 1].strip()
                    full_chunk = f"{header}\n\n{body}"
                    section_title = header.replace("## ", "")
                    all_chunks.append(full_chunk)
                    all_metadatas.append({"source_file": md_file.name, "section": section_title})
                    all_ids.append(f"{md_file.stem}_{i}")
        
        if all_chunks:
            self.collection.add(documents=all_chunks, metadatas=all_metadatas, ids=all_ids)
        return len(all_chunks)
    
    def generate_answer(self, query, category):
        results = self.collection.query(query_texts=[query], n_results=self.top_k)
        chunks = results["documents"][0] if results["documents"] else []
        metadatas = results["metadatas"][0] if results["metadatas"] else []
        distances = results["distances"][0] if results["distances"] else []
        
        if not chunks:
            return {"answer": "I'm not sure about this. A team member will help you shortly.", "confidence": 0.0, "retrieved_chunks": []}
        
        context_parts = []
        for chunk, meta in zip(chunks, metadatas):
            context_parts.append(f"[Source: {meta['source_file']} - {meta['section']}]\n{chunk}")
        context = "\n\n---\n\n".join(context_parts)
        
        system_prompt = f"""You are a professional customer support assistant.

Use the CONTEXT below to answer. Rules:
1. ONLY use context information - never make up
2. If context doesn't have answer: say "I'm not sure, a team member will help you shortly"
3. Warm, professional tone, address customer as "you"
4. Structure: brief greeting + clear answer + next step
5. Maximum 4 sentences
6. NEVER reveal info from "Internal Notes" sections

CONTEXT:
{context}"""
        
        try:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=self.max_tokens,
                system=system_prompt,
                messages=[{"role": "user", "content": query}]
            )
            answer = response.content[0].text.strip()
        except Exception as e:
            logger.error(f"RAG generation error: {e}")
            return {"answer": "Technical issue. A team member will help you shortly.", "confidence": 0.0, "retrieved_chunks": []}
        
        confidence = max(0.0, 1.0 - distances[0]) if distances else 0.0
        retrieved_chunks = [
            {"source": m["source_file"], "section": m["section"], "similarity": 1.0 - d}
            for m, d in zip(metadatas, distances)
        ]
        return {"answer": answer, "confidence": confidence, "retrieved_chunks": retrieved_chunks}


# ============================================================
# ORCHESTRATOR (would be main.py)
# ============================================================

def setup_pipeline():
    escalation_filter = EscalationFilter(keywords_path=ESCALATION_KEYWORDS_PATH)
    classifier = MessageClassifier(
        api_key=ANTHROPIC_API_KEY,
        model=CLASSIFIER_MODEL,
        max_tokens=CLASSIFIER_MAX_TOKENS,
        valid_categories=VALID_CATEGORIES
    )
    rag_pipeline = RAGPipeline(
        kb_directory=KB_DIRECTORY,
        anthropic_api_key=ANTHROPIC_API_KEY,
        model=RESPONDER_MODEL,
        max_tokens=RESPONDER_MAX_TOKENS,
        top_k=TOP_K_RETRIEVAL
    )
    chunks = rag_pipeline.index_knowledge_base()
    logger.info(f"Pipeline ready. {chunks} KB chunks indexed.")
    return escalation_filter, classifier, rag_pipeline


def process_message(message, escalation_filter, classifier, rag_pipeline):
    # Layer 1: Escalation filter
    should_escalate, reason, team = escalation_filter.check(message)
    if should_escalate:
        return {
            "action": "escalate",
            "reason": reason,
            "destination_team": team,
            "response_to_customer": ESCALATION_RESPONSE_TEMPLATE,
            "metadata": {"stage": "escalation_filter"}
        }
    
    # Layer 2: Classifier
    classification = classifier.classify(message)
    if classification["confidence"] < MIN_CONFIDENCE_TO_ANSWER:
        return {
            "action": "escalate",
            "reason": "low_confidence_classification",
            "destination_team": "support_general",
            "response_to_customer": ESCALATION_RESPONSE_TEMPLATE,
            "metadata": {"stage": "classifier", "classification": classification}
        }
    
    # Layer 3: RAG
    rag_result = rag_pipeline.generate_answer(message, classification["category"])
    if rag_result["confidence"] < MIN_CONFIDENCE_TO_ANSWER:
        return {
            "action": "escalate",
            "reason": "low_confidence_rag",
            "destination_team": "support_general",
            "response_to_customer": ESCALATION_RESPONSE_TEMPLATE,
            "metadata": {"stage": "rag", "classification": classification, "rag": rag_result}
        }
    
    return {
        "action": "answer",
        "category": classification["category"],
        "response_to_customer": rag_result["answer"],
        "metadata": {
            "stage": "answered",
            "classification": classification,
            "rag_confidence": rag_result["confidence"],
            "retrieved_chunks": rag_result["retrieved_chunks"]
        }
    }


# Initialize
escalation_filter, classifier, rag_pipeline = setup_pipeline()

## Step 5 — Run test cases

Five messages, each going through a different pipeline path:

In [ ]:
test_messages = [
    "When will my order arrive?",                  # → answer (shipping)
    "I want to refund my order",                   # → escalate (refund keyword)
    "asdfqwerty xyz blah",                         # → escalate (low confidence)
    "How do I download my invoice?",               # → answer (billing)
    "I want to speak to a real person",            # → escalate (human request)
]

for msg in test_messages:
    print("=" * 70)
    print(f"MESSAGE: {msg}")
    result = process_message(msg, escalation_filter, classifier, rag_pipeline)
    print(f"ACTION:  {result['action'].upper()}")
    
    if result["action"] == "escalate":
        print(f"REASON:  {result['reason']}")
        print(f"ROUTE:   {result['destination_team']}")
    else:
        print(f"CATEGORY: {result.get('category', 'N/A')}")
        meta = result.get("metadata", {})
        if "rag_confidence" in meta:
            print(f"CONFIDENCE: {meta['rag_confidence']:.2f}")
    
    print(f"\nRESPONSE:\n{result['response_to_customer']}")
    print()

## Step 6 — Try your own message

Edit the message below and run the cell to test your own scenarios.

In [ ]:
your_message = "I haven't received my order in a week, what do I do?"

result = process_message(your_message, escalation_filter, classifier, rag_pipeline)

print("=" * 70)
print(f"YOUR MESSAGE: {your_message}")
print(f"ACTION: {result['action'].upper()}")
if result['action'] == 'escalate':
    print(f"REASON: {result['reason']}")
    print(f"ROUTE: {result['destination_team']}")
else:
    print(f"CATEGORY: {result.get('category', 'N/A')}")
print(f"\nRESPONSE:\n{result['response_to_customer']}")